## 1. Set Up Project Paths

Import libraries and define the raw and processed data locations.

In [1]:
# import necessary libraries for data cleaning and preprocessing

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

print("=" * 70)
print("HOSPITAL PATIENT ANALYTICS - DATA CLEANING")
print("=" * 70)

# PROJECT PATHS

PROJECT_ROOT = Path.cwd().parent

RAW_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

# Create processed folder if it does not exist
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("\nRaw data folder:") #showing the path of the raw data folder
print(RAW_PATH) # it shows the path of the raw data folder where the original data is stored

print("\nProcessed data folder:")
print(PROCESSED_PATH)

HOSPITAL PATIENT ANALYTICS - DATA CLEANING

Raw data folder:
c:\Users\khala\Project\Hospital-Patient-Analytics\data\raw

Processed data folder:
c:\Users\khala\Project\Hospital-Patient-Analytics\data\processed


## 2. Load and Organize Raw Data

Read the source CSV files and group the DataFrames for shared processing.

In [2]:
# LOAD RAW DATA

patients = pd.read_csv(
    RAW_PATH / "patients.csv"
)

services = pd.read_csv(
    RAW_PATH / "services_weekly.csv"
)

staff = pd.read_csv(
    RAW_PATH / "staff.csv"
)

staff_schedule = pd.read_csv(
    RAW_PATH / "staff_schedule.csv"
)

print("\n✓ All four datasets loaded successfully!")


# CREATE DATASET DICTIONARY

datasets = {
    "patients": patients,
    "services_weekly": services,
    "staff": staff,
    "staff_schedule": staff_schedule
}


✓ All four datasets loaded successfully!


## 3. Inspect Original Dataset Sizes

Review row and column counts before cleaning.

In [3]:
# DISPLAY ORIGINAL DATASET SIZE

print("\n" + "=" * 70)
print("ORIGINAL DATASET SIZE")
print("=" * 70)

for name, df in datasets.items():

    print(
        f"{name:<20} : "
        f"{df.shape[0]:,} rows × {df.shape[1]} columns"
    )


ORIGINAL DATASET SIZE
patients             : 1,000 rows × 7 columns
services_weekly      : 208 rows × 10 columns
staff                : 110 rows × 4 columns
staff_schedule       : 6,552 rows × 6 columns


## 4. Standardize Columns and Text

Normalize column names, trim text fields, and replace empty strings with missing values.

In [ ]:
# STANDARDIZE COLUMN NAMES

def clean_column_names(df):

    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.replace(r"[^a-zA-Z0-9_]", "", regex=True)
    )

    return df


patients = clean_column_names(patients)
services = clean_column_names(services)
staff = clean_column_names(staff)
staff_schedule = clean_column_names(staff_schedule)


print("\n" + "=" * 70)
print("STANDARDIZED COLUMN NAMES")
print("=" * 70)

for name, df in {
    "patients": patients,
    "services_weekly": services,
    "staff": staff,
    "staff_schedule": staff_schedule
}.items():

    print(f"\n{name}:")
    print(df.columns.tolist())

# REMOVE EXTRA SPACES FROM TEXT DATA

def clean_text_columns(df):

    df = df.copy()

    text_columns = df.select_dtypes(
        include=["object"]
    ).columns

    for column in text_columns:

        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
        )

        # Convert empty strings to NaN
        df[column] = df[column].replace(
            "",
            np.nan
        )

    return df


patients = clean_text_columns(patients)
services = clean_text_columns(services)
staff = clean_text_columns(staff)
staff_schedule = clean_text_columns(staff_schedule)


STANDARDIZED COLUMN NAMES

patients:
['patient_id', 'name', 'age', 'arrival_date', 'departure_date', 'service', 'satisfaction']

services_weekly:
['week', 'month', 'service', 'available_beds', 'patients_request', 'patients_admitted', 'patients_refused', 'patient_satisfaction', 'staff_morale', 'event']

staff:
['staff_id', 'staff_name', 'role', 'service']

staff_schedule:
['week', 'staff_id', 'staff_name', 'role', 'service', 'present']


C:\Users\khala\AppData\Local\Temp\ipykernel_25316\2979784117.py:46: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(
C:\Users\khala\AppData\Local\Temp\ipykernel_25316\2979784117.py:46: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.

## 5. Remove Duplicates and Convert Dates

Drop duplicate rows and detect/convert date columns.

In [5]:

# REMOVE DUPLICATE RECORDS

print("\n" + "=" * 70)
print("REMOVING DUPLICATES")
print("=" * 70)

for name, df in {
    "patients": patients,
    "services_weekly": services,
    "staff": staff,
    "staff_schedule": staff_schedule
}.items():

    duplicate_count = df.duplicated().sum()

    print(
        f"{name:<20} : "
        f"{duplicate_count} duplicate rows"
    )

patients = patients.drop_duplicates().reset_index(drop=True)

services = services.drop_duplicates().reset_index(drop=True)

staff = staff.drop_duplicates().reset_index(drop=True)

staff_schedule = (
    staff_schedule
    .drop_duplicates()
    .reset_index(drop=True)
)

# CONVERT DATE COLUMNS

print("\n" + "=" * 70)
print("DATE COLUMN DETECTION")
print("=" * 70)


def convert_date_columns(df):

    df = df.copy()

    converted_columns = []

    for column in df.columns:

        column_name = column.lower()

        if (
            "date" in column_name
            or "datetime" in column_name
            or "timestamp" in column_name
        ):

            original_non_null = df[column].notna().sum()

            converted = pd.to_datetime(
                df[column],
                errors="coerce"
            )

            converted_non_null = converted.notna().sum()

            if (
                original_non_null == 0
                or
                converted_non_null / original_non_null >= 0.80
            ):

                df[column] = converted

                converted_columns.append(column)

    return df, converted_columns


patients, patient_date_columns = (
    convert_date_columns(patients)
)

services, service_date_columns = (
    convert_date_columns(services)
)

staff, staff_date_columns = (
    convert_date_columns(staff)
)

staff_schedule, schedule_date_columns = (
    convert_date_columns(staff_schedule)
)


print("Patients:", patient_date_columns)

print(
    "Services Weekly:",
    service_date_columns
)

print("Staff:", staff_date_columns)

print(
    "Staff Schedule:",
    schedule_date_columns
)


REMOVING DUPLICATES
patients             : 0 duplicate rows
services_weekly      : 0 duplicate rows
staff                : 0 duplicate rows
staff_schedule       : 0 duplicate rows

DATE COLUMN DETECTION
Patients: ['arrival_date', 'departure_date']
Services Weekly: []
Staff: []
Staff Schedule: []


## 6. Calculate Stay Length and Validate Patient Fields

Calculate length of stay and validate age and satisfaction values.

In [6]:
# CALCULATE LENGTH OF STAY

print("\n" + "=" * 70)
print("LENGTH OF STAY")
print("=" * 70)


if (
    "arrival_date" in patients.columns
    and
    "departure_date" in patients.columns
):

    patients["arrival_date"] = pd.to_datetime(
        patients["arrival_date"],
        errors="coerce"
    )

    patients["departure_date"] = pd.to_datetime(
        patients["departure_date"],
        errors="coerce"
    )

    patients["length_of_stay"] = (
        patients["departure_date"]
        -
        patients["arrival_date"]
    ).dt.total_seconds() / 86400

    patients["length_of_stay"] = (
        patients["length_of_stay"]
        .round(2)
    )

    # Negative LOS is invalid
    patients.loc[
        patients["length_of_stay"] < 0,
        "length_of_stay"
    ] = np.nan

    print("✓ Length of stay calculated.")

    print(
        "\nAverage Length of Stay:",
        round(
            patients["length_of_stay"].mean(),
            2
        ),
        "days"
    )

else:

    print(
        "arrival_date or departure_date "
        "was not found."
    )

# VALIDATE AGE

print("\n" + "=" * 70)
print("AGE VALIDATION")
print("=" * 70)


if "age" in patients.columns:

    patients["age"] = pd.to_numeric(
        patients["age"],
        errors="coerce"
    )

    invalid_age = (
        (patients["age"] < 0)
        |
        (patients["age"] > 120)
    )

    print(
        "Invalid age records:",
        invalid_age.sum()
    )

    patients.loc[
        invalid_age,
        "age"
    ] = np.nan

# VALIDATE SATISFACTION

if "satisfaction" in patients.columns:

    patients["satisfaction"] = pd.to_numeric(
        patients["satisfaction"],
        errors="coerce"
    )

    print(
        "\nSatisfaction minimum:",
        patients["satisfaction"].min()
    )

    print(
        "Satisfaction maximum:",
        patients["satisfaction"].max()
    )


LENGTH OF STAY
✓ Length of stay calculated.

Average Length of Stay: 7.41 days

AGE VALIDATION
Invalid age records: 0

Satisfaction minimum: 60
Satisfaction maximum: 99


## 7. Review Missing Values and Final Summary

Check the cleaned datasets for missing cells and remaining duplicates.

In [7]:
# CHECK MISSING VALUES

datasets = {
    "patients": patients,
    "services_weekly": services,
    "staff": staff,
    "staff_schedule": staff_schedule
}


print("\n" + "=" * 70)
print("MISSING VALUES AFTER CLEANING")
print("=" * 70)


for name, df in datasets.items():

    print(f"\n--- {name.upper()} ---")

    missing = df.isnull().sum()

    missing = missing[
        missing > 0
    ]

    if len(missing) == 0:

        print("No missing values.")

    else:

        print(missing)

# FINAL DATASET SUMMARY


print("\n" + "=" * 70)
print("FINAL DATASET SUMMARY")
print("=" * 70)


for name, df in datasets.items():

    print(f"\n{name.upper()}")

    print(
        "Rows:",
        f"{len(df):,}"
    )

    print(
        "Columns:",
        len(df.columns)
    )

    print(
        "Duplicates:",
        df.duplicated().sum()
    )

    print(
        "Missing cells:",
        df.isnull().sum().sum()
    )


MISSING VALUES AFTER CLEANING

--- PATIENTS ---
No missing values.

--- SERVICES_WEEKLY ---
No missing values.

--- STAFF ---
No missing values.

--- STAFF_SCHEDULE ---
No missing values.

FINAL DATASET SUMMARY

PATIENTS
Rows: 1,000
Columns: 8
Duplicates: 0
Missing cells: 0

SERVICES_WEEKLY
Rows: 208
Columns: 10
Duplicates: 0
Missing cells: 0

STAFF
Rows: 110
Columns: 4
Duplicates: 0
Missing cells: 0

STAFF_SCHEDULE
Rows: 6,552
Columns: 6
Duplicates: 0
Missing cells: 0


## 8. Save Cleaned Data

Write the cleaned datasets to the processed-data folder and report completion.

In [ ]:
# SAVE CLEANED DATA

patients.to_csv(
    PROCESSED_PATH / "patients_cleaned.csv",
    index=False
)

services.to_csv(
    PROCESSED_PATH / "services_weekly_cleaned.csv",
    index=False
)

staff.to_csv(
    PROCESSED_PATH / "staff_cleaned.csv",
    index=False
)

staff_schedule.to_csv(
    PROCESSED_PATH / "staff_schedule_cleaned.csv",
    index=False
)

# FINAL MESSAGE

print("\n" + "=" * 70)
print("DATA CLEANING COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nCleaned files created:")

print("✓ patients_cleaned.csv")
print("✓ services_weekly_cleaned.csv")
print("✓ staff_cleaned.csv")
print("✓ staff_schedule_cleaned.csv")

print("\nLocation:")
print(PROCESSED_PATH)

print("\nNext step:")
print("→ Open 03_eda.ipynb")
print("→ Perform Exploratory Data Analysis")
print("→ Create charts and identify hospital patterns")


DATA CLEANING COMPLETED SUCCESSFULLY

Cleaned files created:
✓ patients_cleaned.csv
✓ services_weekly_cleaned.csv
✓ staff_cleaned.csv
✓ staff_schedule_cleaned.csv

Location:
c:\Users\khala\Project\Hospital-Patient-Analytics\data\processed

Next step:
→ Open 03_eda.ipynb
→ Perform Exploratory Data Analysis
→ Create charts and identify hospital patterns
